# Day 3: Hierarchical Models

**Applied Bayesian Statistics Workshop**

---

## Learning Objectives

1. Understand the pooling spectrum: complete, none, and partial
2. Build a hierarchical linear model with varying intercepts
3. Visualize shrinkage
4. Apply non-centered parameterization to improve sampling
5. Compare models with WAIC and LOO
6. Work through a real-world inspired example: educational outcomes across schools

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pymc as pm
import arviz as az
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
az.style.use("arviz-darkgrid")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. The Pooling Spectrum

Suppose we measure test scores in $J$ schools. Three strategies:

| Strategy | Description | Risk |
|---|---|---|
| **Complete pooling** | Ignore school; one global mean | Underfitting |
| **No pooling** | Separate estimate per school | Overfitting (small schools) |
| **Partial pooling** | Hierarchical model: school means are drawn from a shared distribution | Best of both worlds |

$$
\text{Partial pooling: } \quad \mu_j \sim \mathcal{N}(\mu_0, \tau), \quad y_{ij} \sim \mathcal{N}(\mu_j, \sigma)
$$

In [ ]:
# --- Generate grouped data with unequal group sizes ---
n_groups = 8
true_mu_global = 70  # Grand mean test score
true_tau = 8         # Between-school SD
true_sigma = 10      # Within-school SD

# Unequal group sizes (some schools have few students)
group_sizes = np.array([5, 8, 3, 50, 12, 4, 30, 7])
true_group_means = rng.normal(true_mu_global, true_tau, size=n_groups)

school_ids = []
scores = []
for j in range(n_groups):
    school_ids.extend([j] * group_sizes[j])
    scores.extend(rng.normal(true_group_means[j], true_sigma, size=group_sizes[j]))

school_ids = np.array(school_ids)
scores = np.array(scores)

# Compute observed group means
obs_means = np.array([scores[school_ids == j].mean() for j in range(n_groups)])
obs_se = np.array([scores[school_ids == j].std() / np.sqrt(group_sizes[j])
                    for j in range(n_groups)])

print("School | Size | True Mean | Obs Mean")
print("-" * 45)
for j in range(n_groups):
    print(f"  {j:>3}  | {group_sizes[j]:>4} | {true_group_means[j]:>8.1f} | {obs_means[j]:>8.1f}")

In [ ]:
# --- Visual comparison: complete vs no pooling ---
global_mean = scores.mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

colors = plt.cm.tab10(np.arange(n_groups))

# Complete pooling
for j in range(n_groups):
    axes[0].errorbar(j, global_mean, yerr=0, fmt="o", color=colors[j], ms=8)
axes[0].axhline(global_mean, color="k", ls="--", alpha=0.6)
axes[0].set_title("Complete Pooling\n(one estimate for all)")
axes[0].set_xlabel("School")
axes[0].set_ylabel("Score")

# No pooling
for j in range(n_groups):
    axes[1].errorbar(j, obs_means[j], yerr=1.96 * obs_se[j], fmt="o",
                     color=colors[j], ms=8, capsize=4)
axes[1].axhline(global_mean, color="k", ls="--", alpha=0.4)
axes[1].set_title("No Pooling\n(separate per school)")
axes[1].set_xlabel("School")

# True values
for j in range(n_groups):
    axes[2].plot(j, true_group_means[j], "o", color=colors[j], ms=8)
axes[2].axhline(true_mu_global, color="k", ls="--", alpha=0.4)
axes[2].set_title("True Means\n(unknown in practice)")
axes[2].set_xlabel("School")

for ax in axes:
    ax.set_xticks(range(n_groups))

plt.tight_layout()
plt.show()

## 2. Hierarchical Linear Model with Varying Intercepts

$$
\mu_j \sim \mathcal{N}(\mu_0, \tau) \quad \text{(group-level)}
$$
$$
y_{ij} \sim \mathcal{N}(\mu_j, \sigma) \quad \text{(observation-level)}
$$

In [ ]:
# --- Fit hierarchical model (centered parameterization) ---
with pm.Model() as hierarchical_centered:
    # Hyperpriors
    mu_0 = pm.Normal("mu_0", mu=70, sigma=20)
    tau = pm.HalfNormal("tau", sigma=15)

    # Group-level means
    mu_j = pm.Normal("mu_j", mu=mu_0, sigma=tau, shape=n_groups)

    # Observation-level noise
    sigma = pm.HalfNormal("sigma", sigma=15)

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu_j[school_ids], sigma=sigma,
                      observed=scores)

    trace_hier = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        target_accept=0.9, return_inferencedata=True
    )

print(az.summary(trace_hier, var_names=["mu_0", "tau", "sigma"]))

In [ ]:
# --- Group-level estimates ---
print(az.summary(trace_hier, var_names=["mu_j"]))

## 3. Shrinkage Visualization

Partial pooling "shrinks" extreme group estimates toward the grand mean.
Groups with fewer observations are shrunk more.

In [ ]:
# --- Shrinkage plot ---
post_means = az.extract(trace_hier, var_names=["mu_j"])["mu_j"].values.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

for j in range(n_groups):
    # Arrow from no-pooling estimate to partial-pooling estimate
    ax.annotate("",
                xy=(post_means[j], j), xytext=(obs_means[j], j),
                arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))
    ax.plot(obs_means[j], j, "s", color="coral", ms=10, zorder=5)
    ax.plot(post_means[j], j, "o", color="steelblue", ms=10, zorder=5)
    ax.plot(true_group_means[j], j, "D", color="green", ms=8, zorder=5)
    ax.text(obs_means[j] + 0.5, j + 0.2, f"n={group_sizes[j]}", fontsize=9)

ax.axvline(global_mean, color="k", ls="--", alpha=0.5, label="Grand mean")

# Legend
ax.plot([], [], "s", color="coral", ms=10, label="No pooling (observed mean)")
ax.plot([], [], "o", color="steelblue", ms=10, label="Partial pooling (posterior mean)")
ax.plot([], [], "D", color="green", ms=8, label="True mean")

ax.set_yticks(range(n_groups))
ax.set_yticklabels([f"School {j}" for j in range(n_groups)])
ax.set_xlabel("Test Score")
ax.set_title("Shrinkage: No Pooling vs Partial Pooling Estimates")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Quantify shrinkage: RMSE
rmse_no_pool = np.sqrt(np.mean((obs_means - true_group_means)**2))
rmse_partial = np.sqrt(np.mean((post_means - true_group_means)**2))
print(f"RMSE (no pooling):      {rmse_no_pool:.2f}")
print(f"RMSE (partial pooling): {rmse_partial:.2f}")

## 4. Non-Centered Parameterization

The centered parameterization can cause sampling difficulties when `tau` is small
relative to the data (the "funnel" problem). The non-centered reparameterization
breaks the dependency:

**Centered:**
$$\mu_j \sim \mathcal{N}(\mu_0, \tau)$$

**Non-centered:**
$$z_j \sim \mathcal{N}(0, 1), \quad \mu_j = \mu_0 + \tau \cdot z_j$$

Mathematically identical, but much easier for HMC to sample.

In [ ]:
# --- Non-centered hierarchical model ---
with pm.Model() as hierarchical_noncentered:
    # Hyperpriors
    mu_0 = pm.Normal("mu_0", mu=70, sigma=20)
    tau = pm.HalfNormal("tau", sigma=15)

    # Non-centered parameterization
    z_j = pm.Normal("z_j", mu=0, sigma=1, shape=n_groups)
    mu_j = pm.Deterministic("mu_j", mu_0 + tau * z_j)

    # Observation-level noise
    sigma = pm.HalfNormal("sigma", sigma=15)

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu_j[school_ids], sigma=sigma,
                      observed=scores)

    trace_nc = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        target_accept=0.9, return_inferencedata=True
    )

print("--- Non-centered model diagnostics ---")
print(az.summary(trace_nc, var_names=["mu_0", "tau", "sigma"]))

In [ ]:
# --- Compare ESS between centered and non-centered ---
ess_centered = az.ess(trace_hier, var_names=["mu_0", "tau"])
ess_nc = az.ess(trace_nc, var_names=["mu_0", "tau"])

print("Effective Sample Size Comparison:")
print(f"  mu_0 -- Centered: {float(ess_centered['mu_0']):.0f}, "
      f"Non-centered: {float(ess_nc['mu_0']):.0f}")
print(f"  tau  -- Centered: {float(ess_centered['tau']):.0f}, "
      f"Non-centered: {float(ess_nc['tau']):.0f}")

# Check divergences
div_c = int(trace_hier.sample_stats["diverging"].sum().values)
div_nc = int(trace_nc.sample_stats["diverging"].sum().values)
print(f"\nDivergences -- Centered: {div_c}, Non-centered: {div_nc}")

## 5. Model Comparison: WAIC and LOO

We compare three models:
1. **Complete pooling**: single mean for all schools
2. **No pooling**: separate mean per school
3. **Partial pooling**: hierarchical model

Using **LOO-CV** (Leave-One-Out Cross-Validation via Pareto-smoothed importance sampling)
and **WAIC** (Widely Applicable Information Criterion).

In [ ]:
# --- Complete pooling model ---
with pm.Model() as complete_pool:
    mu = pm.Normal("mu", mu=70, sigma=20)
    sigma = pm.HalfNormal("sigma", sigma=15)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=scores)

    trace_cp = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

# --- No pooling model ---
with pm.Model() as no_pool:
    mu_j = pm.Normal("mu_j", mu=70, sigma=20, shape=n_groups)
    sigma = pm.HalfNormal("sigma", sigma=15)
    y_obs = pm.Normal("y_obs", mu=mu_j[school_ids], sigma=sigma,
                      observed=scores)

    trace_np = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print("All models fitted.")

In [ ]:
# --- Compute LOO and WAIC for all three models ---
# Add log-likelihood to the traces
with complete_pool:
    pm.compute_log_likelihood(trace_cp)
with no_pool:
    pm.compute_log_likelihood(trace_np)
with hierarchical_noncentered:
    pm.compute_log_likelihood(trace_nc)

# LOO comparison
loo_cp = az.loo(trace_cp)
loo_np = az.loo(trace_np)
loo_pp = az.loo(trace_nc)

comparison = az.compare(
    {"Complete pooling": trace_cp,
     "No pooling": trace_np,
     "Partial pooling": trace_nc}
)

print(comparison)

In [ ]:
# --- Visual model comparison ---
az.plot_compare(comparison)
plt.title("Model Comparison (LOO-CV)")
plt.tight_layout()
plt.show()

## 6. Real-World Example: Educational Outcomes Across Schools

We extend the model to include a school-level predictor (e.g., per-student
funding) and student-level predictor (e.g., study hours).

$$
y_{ij} = \alpha_j + \beta \cdot \text{hours}_{ij} + \varepsilon_{ij}
$$
$$
\alpha_j = \gamma_0 + \gamma_1 \cdot \text{funding}_j + u_j, \quad u_j \sim \mathcal{N}(0, \tau)
$$

In [ ]:
# --- Generate richer synthetic data ---
n_schools = 10
students_per_school = rng.integers(10, 50, size=n_schools)

# School-level predictor: per-student funding (standardized)
funding = rng.normal(0, 1, size=n_schools)

# True parameters
gamma_0_true = 65.0   # Grand intercept
gamma_1_true = 5.0    # Effect of funding on school intercept
beta_true = 3.0       # Effect of study hours
tau_true = 4.0        # Between-school SD (residual)
sigma_true = 8.0      # Within-school SD

# Generate group-level random effects
u_j_true = rng.normal(0, tau_true, size=n_schools)
alpha_j_true = gamma_0_true + gamma_1_true * funding + u_j_true

# Generate student-level data
school_idx = []
hours = []
scores_edu = []

for j in range(n_schools):
    nj = students_per_school[j]
    school_idx.extend([j] * nj)
    h = rng.normal(5, 2, size=nj).clip(0, 15)  # Study hours
    hours.extend(h)
    y_ij = alpha_j_true[j] + beta_true * h + rng.normal(0, sigma_true, size=nj)
    scores_edu.extend(y_ij)

school_idx = np.array(school_idx)
hours = np.array(hours)
scores_edu = np.array(scores_edu)

print(f"Total students: {len(scores_edu)}")
print(f"Schools: {n_schools}")
print(f"Students per school: {students_per_school}")

In [ ]:
# --- Visualize data by school ---
fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.tab10(np.arange(n_schools))
for j in range(n_schools):
    mask = school_idx == j
    ax.scatter(hours[mask], scores_edu[mask], color=colors[j], alpha=0.4,
              s=20, label=f"School {j} (n={students_per_school[j]})")

ax.set_xlabel("Study Hours")
ax.set_ylabel("Test Score")
ax.set_title("Educational Outcomes: Study Hours vs Test Scores by School")
ax.legend(fontsize=8, ncol=2, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit the full hierarchical model ---
with pm.Model() as edu_model:
    # Hyperpriors
    gamma_0 = pm.Normal("gamma_0", mu=65, sigma=20)
    gamma_1 = pm.Normal("gamma_1", mu=0, sigma=10)
    tau = pm.HalfNormal("tau", sigma=10)

    # Non-centered school random effects
    z_j = pm.Normal("z_j", mu=0, sigma=1, shape=n_schools)
    alpha_j = pm.Deterministic("alpha_j",
                                gamma_0 + gamma_1 * funding + tau * z_j)

    # Student-level slope
    beta = pm.Normal("beta", mu=0, sigma=10)

    # Observation noise
    sigma = pm.HalfNormal("sigma", sigma=15)

    # Expected value
    mu = alpha_j[school_idx] + beta * hours

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=scores_edu)

    trace_edu = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        target_accept=0.9, return_inferencedata=True
    )

print(az.summary(trace_edu,
                 var_names=["gamma_0", "gamma_1", "beta", "tau", "sigma"]))

In [ ]:
# --- Compare estimated school intercepts with true values ---
alpha_post = az.extract(trace_edu, var_names=["alpha_j"])["alpha_j"].values
alpha_post_means = alpha_post.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(alpha_j_true, alpha_post_means, s=80, zorder=5,
           edgecolors="k", linewidths=0.5)

# Add error bars (94% HDI)
for j in range(n_schools):
    lo, hi = np.percentile(alpha_post[j], [3, 97])
    ax.plot([alpha_j_true[j], alpha_j_true[j]], [lo, hi],
            "k-", lw=1, alpha=0.5)
    ax.text(alpha_j_true[j] + 0.3, alpha_post_means[j] + 0.3,
            f"{j}", fontsize=9)

# 45-degree line
lim = [min(alpha_j_true.min(), alpha_post_means.min()) - 3,
       max(alpha_j_true.max(), alpha_post_means.max()) + 3]
ax.plot(lim, lim, "r--", lw=1.5, label="Perfect recovery")

ax.set_xlabel("True School Intercept")
ax.set_ylabel("Posterior Mean Intercept")
ax.set_title("Parameter Recovery: School Intercepts")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior distributions of key parameters ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

param_info = [
    ("gamma_0", gamma_0_true, "Grand intercept"),
    ("gamma_1", gamma_1_true, "Funding effect"),
    ("beta", beta_true, "Study hours effect"),
    ("sigma", sigma_true, "Within-school SD"),
]

for ax, (var, true_val, label) in zip(axes.ravel(), param_info):
    samples = az.extract(trace_edu, var_names=[var])[var].values
    ax.hist(samples, bins=50, density=True, alpha=0.6, color="steelblue",
            edgecolor="white")
    ax.axvline(true_val, color="red", ls="--", lw=2,
               label=f"True: {true_val}")
    ax.axvline(samples.mean(), color="orange", lw=2,
               label=f"Post. mean: {samples.mean():.2f}")
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.suptitle("Posterior Distributions of Key Parameters", fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Partial pooling** (hierarchical models) balances bias and variance by sharing information across groups.
2. **Shrinkage** pulls extreme estimates from small groups toward the grand mean -- this is a feature, not a bug.
3. **Non-centered parameterization** often improves MCMC sampling by removing the funnel geometry.
4. **LOO-CV and WAIC** provide principled ways to compare models on predictive accuracy.
5. Hierarchical models naturally handle **unbalanced data** (different group sizes).

---

*Next: Day 4 -- Advanced Applications*